In [1]:
!pip install openai pandas

In [2]:
!pip install python-dotenv

In [3]:
import json
from openai import OpenAI
import pandas as pd
from IPython.display import Image, display

In [4]:
import os
import logging
logging.basicConfig(
    format='%(asctime)s : %(levelname)s - %(message)s',
    level=logging.INFO
)
from dotenv import load_dotenv
load_dotenv()


True

In [5]:
OPENAI_API_KEY=os.getenv('OPENAI_API_KEY')

In [21]:
client=OpenAI(
    api_key=OPENAI_API_KEY
)



In [22]:
response=client.responses.create(
    model='gpt-5-mini',
    input='Write one sentence bedtime story about Korean 2015 drama series Reply 1988'
)
print(response.output_text)

2025-11-24 11:04:34,388 : INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


As the moon rose over Ssangmun-dong, five lifelong friends and their families drifted off surrounded by laughter, home-cooked scents, and the gentle promise that the warmth of their neighborhood would keep their sweetest memories safe forever.


In [7]:
df=pd.read_csv('/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/original/Catalog_mapping.csv')
df.head()

,dcterms:title,dcterms:creator,dcterms:description,dcatin:jurisdictionLevel,dcterms:created,dcterms:issued,dcterms:modified,dcat:accessURL,dcterms:identifier,dcat:landingPage,...,dcterms:isPartOf,dcterms:isReferencedBy,dcterms:isReplacedBy,dcterms:replaces,dcterms:isVersionOf,dcterms:hasVersion,dcterms:relation,Unnamed: 43,No mapping available,No mapping available.1
0,All India Pincode Boundary Geo JSON,Ministry of Communications;Department of Posts,This will provide the Pincode boundary of Post...,Central,2025-05-09,2025-05-09,2025-05-09,NaN,NaN,https://data.gov.in/catalog/all-india-pincode-...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,923",0
1,Crime in India - 2022,Ministry of Home Affairs;Department of States;...,Catalog contains data of Crime in India - 2022...,Central,2024-04-10,2024-04-12,2024-04-12,NaN,NaN,https://data.gov.in/catalog/crime-india-2022,...,NaN,NaN,NaN,NaN,Crime in India [https://www.data.gov.in/datase...,NaN,NaN,NaN,"32,297",0
2,Indian Sign Language Dictionary,Ministry of Social Justice and Empowerment;Dep...,The Indian Sign Language (ISL) Dictionary has ...,Central,2024-02-06,2024-02-07,2025-02-18,NaN,NaN,https://data.gov.in/catalog/indian-sign-langua...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"11,580",0
3,Demographic Data of Unorganised Workers regist...,Ministry of Labour and Employment,The catalog contains data for self-registratio...,Central,2023-06-07,2023-12-21,2025-06-19,NaN,NaN,https://data.gov.in/catalog/demographic-data-u...,...,https://www.data.gov.in/catalog/eshram-registr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"51,289",0
4,Answers Data of Rajya Sabha Questions for Sess...,Rajya Sabha,Get questions' Answers Data of Session 260 of ...,Central,2023-12-18,2023-12-19,2024-03-11,NaN,NaN,https://data.gov.in/catalog/answers-data-rajya...,...,NaN,NaN,NaN,NaN,Rajya Sabha Questions: https://www.data.gov.in...,NaN,NaN,NaN,"4,514",0


In [8]:
#for filling in title, description, note wherever required.
# TODO Create a generic prompt template 
metadata_enhancement_prompt="""

# System Prompt: Catalog-Level Metadata Enhancement for India Government Open Data

## Role
You are an expert metadata curator specializing in government open data standards including Dublin Core, DCAT v3, and India's data.gov.in cataloging specifications. Your task is to generate enhanced, standardized metadata **for catalog-level metadata**, focusing specifically on improving:
- **dcterms:title** (Catalog Title)
- **dcterms:description** (Catalog Description)

Catalogs typically do **not** contain short description, abstract, or notes like dataset resources. Therefore, your focus is to create **rich, clear, comprehensive catalog metadata** that improves discoverability and comprehension without inventing facts not supported by metadata.

## Instructions
Do not make mistakes.  
Keep all enhancements **catalog-level**, not dataset-level.  
Do not add fictitious indicators, metrics, units, or field names.

## Input Format
You will receive CSV rows with the following key columns:
- **dcterms:title** – Existing catalog title (often short, incomplete, or unclear)
- **dcterms:description** – Current description (may be empty or minimal)
- **dcat:theme** – Thematic categories (semicolon-separated)
- **dcterms:spatial** – Geographic scope (e.g., India, specific states)
- **dcatin:jurisdictionLevel** – Government level (Central/State)
- **dct:accrualPeriodicity** – Update frequency (Daily, Monthly, Annual, etc.)
- **dcat:keyword** – Existing keywords
- **dcterms:publisher** – Publishing ministry/department
- Other administrative fields for context

---

# Task Requirements

## 1. Generate Enhanced Catalog Title (dcterms:title)
Create an improved catalog title that:
- Preserves the original meaning
- Adds clarity, expands abbreviations, and removes ambiguity
- Clearly indicates domain and scope
- Is formatted in Title Case
- Length: **10–20 words**, maximum 25
- Avoids dataset-level terminology like “records”, “fields”, “table”, “resource”

### Enhancement Logic
- Start with existing title → rewrite for clarity
- Expand abbreviations (RoC → Registrars of Companies)
- Add context if the title is too vague
- Include geographic or temporal scope where clearly implied
- Do **not** fabricate specifics (e.g., number of datasets)

---

## 2. Generate Comprehensive Catalog Description (dcterms:description)
Write a **2–4 sentence** description that:
- Explains what the catalog contains (in general terms)
- Mentions thematic scope (agriculture, health, finance, etc.)
- Explains purpose and relevance of the catalog
- Mentions update frequency and jurisdiction if available
- Provides high-level context without dataset-specific field details
- Length: **70–150 words**

### Generation Logic
- If description exists → enhance, expand, modernize
- If empty → generate from available metadata fields
- Mention:
  - What the catalog covers
  - Its purpose or utility
  - Geographic scope (India, states)
  - Update frequency if relevant
  - Publisher or agency context

### Strict Constraints
- **Do not mention specific dataset fields or column names**
- **Do not invent methodology details**
- **Do not generate dataset-level descriptions**
- Keep content catalog-level and generalized

---

# Output Format
Return valid JSON with:

{
  "enhanced_title": "...",
  "enhanced_description": "...",
  "processing_summary": {
    "title_changes": "Brief explanation of improvements",
    "description_changes": "How the description was expanded/clarified"
  },
  "metadata_quality_score": 0.85,
  "quality_score_explanation": "How score was calculated",
  "source_fields_used": ["list", "of", "fields", "used"]
}

### Metadata Quality Score (0.0 to 1.0)
- Completeness (50%): Title + description enhanced
- Enhancement Quality (30%): Degree of clarity added
- Input Metadata Usefulness (20%): How much metadata contributed

Scoring guidelines:
- **0.9–1.0**: Strong enhancements and rich metadata
- **0.7–0.89**: Good enhancements, moderate metadata
- **0.5–0.69**: Basic enhancements, low metadata richness
- **<0.5**: Very limited possible improvement

---

## Special Handling for Indian Government Catalogs

### Ministry/Department Context
- Expand ministry names in description
- Use formal names in full (Ministry of Agriculture & Farmers Welfare)

### Geographic Indicators
- Use “India” or state names explicitly
- Include jurisdiction level where helpful

### Sector-Specific Notes (high level only)
- Agriculture → markets, crops, production, prices
- Health → indicators, public health programs
- Finance → economic indicators, annual trends
- Education → learning, schools, enrolment

### Acronym Handling
Expand common acronyms at catalog level:
- KCC → Kisan Call Centre  
- APMC → Agricultural Produce Market Committee  
- PHC → Primary Health Centre  
- MSME → Micro, Small & Medium Enterprises  
- RoC → Registrars of Companies  

---

## Error Handling
- Missing title → Score = 0.0; return explanation
- Empty description → Generate new based on other fields
- Very low metadata → Provide clean generic description with low score

## Constraints
- Maintain factual accuracy
- Use Indian English
- Do not add dataset-level specifics
- Keep language professional, clear, and consistent

"""


In [25]:
# for keyword/theme generation
categorize_system_prompt="""
# SYSTEM PROMPT: Catalog-Level Keyword, Sector (Theme) and Group (Subject) Enhancement

## ROLE
You are an expert metadata curator specializing in Indian Open Government Data (OGD), DCAT-India, Dublin Core, and metadata taxonomy design.  
Your task is to generate clean, consistent, catalog-level:
- **dcat:keyword** (Layman Keywords)
- **dcat:theme** (Sector)

based on the catalog's high-level purpose, using the given inputs.

This enhancement is for **CATALOGS**, not datasets.

---

# INPUT FORMAT
You will receive the following fields:

- **dcterms:subject** → existing Groups (may be empty, noisy, inconsistent)  
- **dcat:theme** → existing Sectors (may be empty or too generic)  
- **dcat:keyword** → existing Keywords (often sparse or unclean)
- **dcterms:title** – Existing catalog title (often short, incomplete, or unclear)
- **dcterms:description** – Current description (may be empty or minimal)

Use these fields + the additional context provided (title, description, publisher, etc.) to generate **new enriched metadata values**.

---

# TASK REQUIREMENTS
## 1. Enhanced Layman Keywords (`enhanced_keywords`)

Generate **6–10 highly discoverable**, layman-friendly keywords.

### Rules
- Must be **1–2 words**, in **CamelCase**.  
- Must reflect the **actual content** of the dataset.  
- Should help a normal user find this dataset easily.  
- Avoid:
  - Ministry/Department names  
  - Administrative terms  
  - Very generic umbrella words  
  - The word *“India”*  
  - Technical/internal vocabulary  
- Use the dataset **Title, description, Sector, etc. 




---

## 2. Sponsored Keywords (`sponsored_keywords`)

These are the **3–5 strongest search terms**, directly tied to the dataset **title**.

### Rules
- 1–2 words, **CamelCase**.  
- Highest semantic similarity to the dataset title.  
- Avoid administrative terms, avoid “India” unless in the official title.
- Should be slighly different from `enhanced_keywords`, focusing more on exact title relevance.
- Avoid making them identical to `enhanced_keywords`.


---

## 3. Generate Sector (dcat:theme)
One or more sectors may be allowed, but **prioritize ONE primary sector**.

Choose from controlled vocabulary that is used by dcat-AP:

- Agriculture  
- Health  
- Education  
- Finance & Economy  
- Transport  
- Energy  
- Environment  
- Governance & Administration  
- Social Welfare  
- Rural Development  
- Urban Development  
- Water Resources  
- Disaster Management  
- Science & Technology  
- Commerce & Industry  
- Housing  
- Employment & Labour  
- Others (only when unavoidable)

Rules:
- Sector must reflect the **overall domain** of the entire catalog
- Use publisher/ministry to infer if missing
- Do not assign overly broad sector (“Others” only if no other fits)

---

# OUTPUT FORMAT

Return a **valid JSON object**:

{
  "generated_keywords": ["keyword1", "keyword2", ...],
  "sponsored_keywords": ["keyword1", "keyword2", ...],
  "sector": "Primary sector assigned",
  "processing_summary": {
    "keyword_logic": "How keywords were derived",
    "sector_logic": "Why the sector was selected",
    "sponsored_keywords": "How sponsored keywords were chosen"
  },
  "source_fields_used": ["dcterms:subject", "dcat:theme", "dcat:keyword"]
}

---

# SPECIAL HANDLING FOR INDIAN GOVERNMENT CONTEXT
- Use official terminology aligned with Indian ministries
- Abbreviations allowed only if widely used (GST, APMC, PHC, NFHS)
- Prefer Indian-specific terms (district, state, block, panchayat)
- Ensure classification reflects India’s administrative domains

---

# ERROR HANDLING
- If all inputs are empty → generate generic but safe output  
- If conflicts exist → trust catalog title > description > publisher > input fields  
- If contradictory themes exist → choose the most central domain
"""


In [ ]:

# for keyword generation
keyword_prompt= '''




## Metadata Keyword Generation + High Value Data (HVD) Classification Prompt

Role: You are an expert metadata curator specializing in government open data standards (Dublin Core, DCAT v3, data.gov.in). Your task is to generate **layman-friendly keywords**, **sponsored keywords**, and **High Value Data (HVD) category** for a dataset using the metadata provided.

You will receive the dataset metadata in plain text, for example:

Title: <title text>  
Catalog Title: <catalog_title text>  
Ministry/Department: <ministry_department text>  
Sector: <sector text>  
Sector Resource: <sector_resource text>  
CDOS State Ministry: <cdos_state_ministry text>  
Note: <note or description text>  


Your job is to analyze this metadata and return **one single JSON object** following the structure described below.

---

# 1. Enhanced Layman Keywords (`enhanced_keywords`)

Generate **6–10 highly discoverable**, layman-friendly keywords.

### Rules
- Must be **1–2 words**, in **CamelCase**.  
- Must reflect the **actual content** of the dataset.  
- Should help a normal user find this dataset easily.  
- Avoid:
  - Ministry/Department names  
  - Administrative terms  
  - Very generic umbrella words  
  - The word *“India”*  
  - Technical/internal vocabulary  
- Use the dataset **Title, Catalog Title, Sector, Sector Resource, Note**.

### Output format
A **comma-separated string**, e.g.:

```

"enhanced_keywords": "RainfallData, WeatherTrend, DailyRain, MonsoonPattern"

```

---

# 2. Sponsored Keywords (`sponsored_keywords`)

These are the **3–5 strongest search terms**, directly tied to the dataset **title**.

### Rules
- 1–2 words, **CamelCase**.  
- Highest semantic similarity to the dataset title.  
- Avoid administrative terms, avoid “India” unless in the official title.
- Should be slighly different from `enhanced_keywords`, focusing more on exact title relevance.
### Output format

```

"sponsored_keywords": "MandiPrice, CommodityRate, MarketPrice"

```

---

# 3. High Value Data Classification (`hvd_category`)

You will receive:

`HVD Flag: <0 or 1>`

### If HVD Flag = 0:
- No classification.
- Output:

- You must **always** generate `enhanced_keywords`, `sponsored_keywords`, `justification`, `confidence_score`, and `metadata_gaps` **for every dataset**, **regardless of the HVD Flag value**.  
- The HVD Flag only affects the value of `"hvd_category"`.

```

"hvd_category": ""

```

### If HVD Flag = 1:
Classify the dataset into **one** category from:

1. geospatial  
2. earth observation and environment  
3. meteorological  
4. statistics  
5. companies and company ownership  
6. mobility  

Choose based on **Title, Catalog Title, Note**, and content meaning.

### Output format example
```

"hvd_category": "statistics"

```

---

# 4. Additional Required Fields

Your JSON must also include:

| Key | Description |
|------|------------|
| `"title"` | Copy the dataset title from the input |
| `"justification"` | 1–2 sentences explaining keyword + HVD reasoning |
| `"confidence_score"` | Float 0–1 indicating confidence |
| `"metadata_gaps"` | List describing missing or unclear metadata |

If no metadata gaps:



"metadata_gaps": []



---

# 5. Output Format (STRICT)

You must output **exactly ONE JSON object**, with **ALL keys**, and **nothing else**.

- No markdown  
- No comments  
- No explanations outside JSON  
- No "error" field  
- Must be valid JSON  

### Required JSON structure:

json
{
  "title": "",
  "enhanced_keywords": "",
  "sponsored_keywords": "",
  "justification": "",
  "confidence_score": 0.0,
  "metadata_gaps": [],
  "hvd_category": ""
}


Populate every field appropriately.

---

# 6. Example Output (Structure Only)

json
{
  "title": "Current Daily Price of Various Commodities from Various Markets (Mandi)",
  "enhanced_keywords": "MandiPrice, CommodityPrice, DailyPrice, MarketRate, WholesalePrice, AgricultureMarket, PriceTrend, CropPrice",
  "sponsored_keywords": "MandiPrice, CommodityPrice, MarketRate, DailyPrice",
  "justification": "The dataset provides daily wholesale commodity prices, so keywords focus on commodity pricing and mandis. It fits the statistics HVD category because it involves structured numerical market data.",
  "confidence_score": 0.9,
  "metadata_gaps": ["Geographic coverage missing", "Temporal start date not specified"],
  "hvd_category": "statistics"
}









'''

In [26]:
def get_keywords(metadata_content):
    response=client.chat.completions.create(
         model='gpt-5-nano',
         temperature=1,
         response_format={
             "type":"json_object",
        
         },
         messages=[
              {
                  "role":"system",
                  "content":categorize_system_prompt
              },
              {
                  "role":"user",
                  "content":metadata_content
              }
         ],

    )
    return json.loads(response.choices[0].message.content)

In [9]:
def get_enhanced_metadata(metadata_content):
    response = client.chat.completions.create(
        model='gpt-5-nano',
        temperature=1,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": metadata_enhancement_prompt
            },
            {
                "role": "user",
                "content": metadata_content
            }
        ],
    )
    return json.loads(response.choices[0].message.content)

In [10]:
df=pd.read_csv("/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/ogd_metadata_sample/sample_catalog_nic.csv", encoding="cp1252", )
df.head()

,title,body:value,changed,created,published_date,field_asset_jurisdiction:name,field_ds_govt_type,field_ministry_department:name,field_sector:name,from_api,keywords,high_value_dataset,is_api_available,node_alias,ogdp_view_count
0,All India Pincode Boundary Geo JSON,This will provide the Pincode boundary of Post...,5/9/2025,5/9/2025,5/9/2025,All India,Central,Ministry of Communications;Department of Posts,All,0,PINCODE;Geojson ;Geo Json File,1,0,/catalog/all-india-pincode-boundary-geo-json,5
1,Crime in India - 2022,Catalog contains data of Crime in India - 2022...,4/12/2024,4/10/2024,4/12/2024,All India,Central,Ministry of Home Affairs;Department of States;...,Police,0,Indian Penal Code;Crime;Crimes under Special a...,0,0,/catalog/crime-india-2022,4251
2,Indian Sign Language Dictionary,The Indian Sign Language (ISL) Dictionary has ...,2/18/2025,2/6/2024,2/7/2024,All India,Central,Ministry of Social Justice and Empowerment;Dep...,Disabled,0,Sign Language;Indian Sign Language;Sign Langua...,0,0,/catalog/indian-sign-language-dictionary,3268
3,Demographic Data of Unorganised Workers regist...,The catalog contains data for self-registratio...,6/19/2025,6/7/2023,12/21/2023,All India,Central,Ministry of Labour and Employment,Unorganized Sector Workers,1,eShram;eShram Registration;MoLE;Labour and Emp...,1,0,/catalog/demographic-data-unorganised-workers-...,21424
4,Answers Data of Rajya Sabha Questions for Sess...,Get questions' Answers Data of Session 260 of ...,3/11/2024,12/18/2023,12/19/2023,All India,Central,Rajya Sabha,All,0,Rajya Sabha,0,0,/catalog/answers-data-rajya-sabha-questions-se...,2811


In [14]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
from threading import Lock

In [15]:
def extract_clean_value(value):
    """Clean and extract value from field"""
    if pd.isna(value) or value == 'nan' or value == '':
        return ""
    return str(value).strip()

def parse_semicolon_separated(value):
    """Parse semicolon-separated values into a list"""
    if not value or pd.isna(value):
        return []
    return [v.strip() for v in str(value).split(';') if v.strip()]

In [12]:
def process_row_generation(row):
    """Process a single catalog row for title + description enhancement only"""

    # --- helper to robustly read columns (handles leading/trailing spaces in sheet) ---
    def safe_get(key, default=""):
        if key in row:
            return row.get(key, default)
        # fallback: match by stripped key
        for k in row.index:
            if isinstance(k, str) and k.strip() == key:
                return row.get(k, default)
        return default
        
    # Primary fields for enhancement (catalog-level)
    title = extract_clean_value(safe_get("dcterms:title", ""))
    description = extract_clean_value(safe_get("dcterms:description", ""))

    # Supporting metadata fields (based on actual sheet columns)
    themes = parse_semicolon_separated(safe_get("dcat:theme", ""))
    spatial = extract_clean_value(safe_get("dcterms:spatial", ""))
    temporal = extract_clean_value(safe_get("dcterms:temporal", ""))
    jurisdiction = extract_clean_value(safe_get("dcatin:jurisdictionLevel", ""))
    frequency = extract_clean_value(safe_get("dct:accrualPeriodicity", ""))
    keywords = extract_clean_value(safe_get("dcat:keyword", ""))
    publisher = extract_clean_value(safe_get("dcterms:publisher", ""))
    creator = extract_clean_value(safe_get("dcterms:creator", ""))

    # vCard fields in your sheet have leading spaces, safe_get handles that
    contact_name = extract_clean_value(safe_get("vCard:fn", ""))
    org_name = extract_clean_value(safe_get("vcard:organization-name", ""))
    contact_email = extract_clean_value(safe_get("vcard:hasEmail", ""))
    contact_role = extract_clean_value(safe_get("vcard:role", ""))

    # Build metadata content string for the LLM (catalog-level)
    metadata_content = f'''
=== CURRENT CATALOG METADATA ===
Catalog Title: {title}
Catalog Description: {description}

=== CONTEXTUAL INFORMATION ===
Themes: {'; '.join(themes) if themes else 'Not specified'}
Keywords: {keywords if keywords else 'Not specified'}
Publisher: {publisher if publisher else 'Not specified'}
Creator: {creator if creator else 'Not specified'}
Organization: {org_name if org_name else 'Not specified'}
Contact Person: {contact_name if contact_name else 'Not specified'}
Contact Email: {contact_email if contact_email else 'Not specified'}
Contact Role: {contact_role if contact_role else 'Not specified'}

=== COVERAGE DETAILS ===
Spatial Coverage: {spatial if spatial else 'India'}
Temporal Coverage: {temporal if temporal else 'Not specified'}
Jurisdiction Level: {jurisdiction if jurisdiction else 'Not specified'}

=== UPDATE DETAILS ===
Update Frequency: {frequency if frequency else 'Not specified'}

TASK: Generate enhanced catalog-level metadata values for dcterms:title and dcterms:description only, according to the system prompt.
'''

    try:
        result = get_enhanced_metadata(metadata_content)
        logging.info(f"Processed catalog: {title[:50]}...")

    except Exception as e:
        logging.error(f"LLM failed for catalog '{title}': {e}")
        result = {
            "enhanced_title": title,
            "enhanced_description": description,
            "processing_summary": {
                "title_changes": "",
                "description_changes": "",
                "error": str(e)
            },
            "metadata_quality_score": 0.0,
            "quality_score_explanation": f"Processing failed: {str(e)}",
            "source_fields_used": []
        }

    return {
        "original_title": str(title),
        "original_description": str(description)[:500],

        # Enhanced catalog outputs
        "enhanced_title": result.get("enhanced_title", title),
        "enhanced_description": result.get("enhanced_description", description),

        # Summary + scoring
        "processing_summary": result.get("processing_summary", {}),
        "quality_score": result.get("metadata_quality_score", 0.0),
        "quality_score_explanation": result.get("quality_score_explanation", ""),
        "source_fields_used": result.get("source_fields_used", []),

        # Context fields retained for audit/debug
        "themes": str('; '.join(themes) if themes else ''),
        "publisher": str(publisher),
        "jurisdiction": str(jurisdiction),
        "frequency": str(frequency),

        # Full LLM JSON for traceability
        "llm_response": json.dumps(result)
    }


In [23]:
# Main execution
input_file = "/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/original/Catalog_mapping.csv"
output_file = "/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/metadata_cleaned_catalog_title_description.csv"

df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows")

csv_lock = Lock()

# Only fields that exist in the new function + new prompt
fieldnames = [
    # Original fields
    "original_title",
    "original_description",

    # Enhanced fields
    "enhanced_title",
    "enhanced_description",

    # Context fields (kept for audit/debug)
    "themes",
    "publisher",
    "jurisdiction",
    "frequency",

    # Processing metadata
    "quality_score",
    "quality_score_explanation",
    "title_changes",
    "description_changes",
    "source_fields_used",
    "llm_response"
]

# Write header once
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

results_array = []
max_workers = 8
rows_to_process = df.head(105)  # Change to df for all rows

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_generation, row): idx
        for idx, row in rows_to_process.iterrows()
    }

    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()

            processing_summary = result_obj.get("processing_summary", {})

            csv_row = {
                # Original fields
                "original_title": result_obj.get("original_title", ""),
                "original_description": result_obj.get("original_description", ""),

                # Enhanced fields
                "enhanced_title": result_obj.get("enhanced_title", ""),
                "enhanced_description": result_obj.get("enhanced_description", ""),

                # Context fields
                "themes": result_obj.get("themes", ""),
                "publisher": result_obj.get("publisher", ""),
                "jurisdiction": result_obj.get("jurisdiction", ""),
                "frequency": result_obj.get("frequency", ""),

                # Processing metadata
                "quality_score": result_obj.get("quality_score", 0.0),
                "quality_score_explanation": result_obj.get("quality_score_explanation", ""),
                "title_changes": processing_summary.get("title_changes", ""),
                "description_changes": processing_summary.get("description_changes", ""),
                "source_fields_used": json.dumps(result_obj.get("source_fields_used", [])),
                "llm_response": result_obj.get("llm_response", "")
            }

            results_array.append(csv_row)

            with csv_lock:
                with open(output_file, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(csv_row)

            print(f"✓ {csv_row['original_title'][:60]}...")

        except Exception as e:
            logging.error(f"Row failed: {e}")

print(f"\n✓ Complete! Results saved to {output_file}")


Loaded 86 rows


2025-11-24 11:05:34,311 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:05:34,432 : INFO - Processed catalog: Indian Sign Language Dictionary...


✓ Indian Sign Language Dictionary...


2025-11-24 11:05:35,897 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:05:35,904 : INFO - Processed catalog: Answers Data of Rajya Sabha Questions for Session ...


✓ Answers Data of Rajya Sabha Questions for Session 260...


...

✓ District Rainfall Normal (in mm) Monthly, Seasonal And Annua...


2025-11-24 11:12:04,197 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:12:04,203 : INFO - Processed catalog: Quarterly Estimates of GDP at Constant Prices...


✓ Quarterly Estimates of GDP at Constant Prices...


2025-11-24 11:12:11,614 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:12:11,618 : INFO - Processed catalog: District-wise, season-wise crop production statist...


✓ District-wise, season-wise crop production statistics...

✓ Complete! Results saved to /home/aakash/NIC/cdl-updated/nic-metadata-cleaning/data/metadata_cleaned_catalog_title_description.csv


In [27]:
def process_row_keyword(row):
    """Process a single catalog row - thread-safe, aligned with catalog prompt"""

    # --- helper to robustly read columns (handles leading/trailing spaces) ---
    def safe_get(key, default=""):
        if key in row:
            return row.get(key, default)
        for k in row.index:
            if isinstance(k, str) and k.strip() == key:
                return row.get(k, default)
        return default

    # Primary catalog fields
    title = extract_clean_value(safe_get("dcterms:title", ""))
    description = extract_clean_value(safe_get("dcterms:description", ""))
    subject = extract_clean_value(safe_get("dcterms:subject", ""))   # mapped to groups
    theme = extract_clean_value(safe_get("dcat:theme", ""))          # mapped to sectors
    keywords = extract_clean_value(safe_get("dcat:keyword", ""))     # existing keywords

    # Build metadata content string for LLM
    metadata_content = f'''
Catalog Title: {title}
Catalog Description: {description}

Existing Groups (dcterms:subject): {subject if subject else "Not specified"}
Existing Sectors (dcat:theme): {theme if theme else "Not specified"}
Existing Keywords (dcat:keyword): {keywords if keywords else "Not specified"}

TASK: Generate enhanced_keywords, sponsored_keywords, and primary sector for this catalog,
as per the system prompt. Output valid JSON only.
'''

    try:
        result = get_keywords(metadata_content)  # expects JSON per new prompt
        logging.info(f"Result is {result}")

    except Exception as e:
        logging.error(f"LLM failed for '{title}': {e}")
        result = {
            "generated_keywords": [],
            "sponsored_keywords": [],
            "sector": "Others",
            "processing_summary": {
                "keyword_logic": "",
                "sector_logic": "",
                "sponsored_keywords": "",
                "error": str(e)
            },
            "source_fields_used": []
        }

    processing_summary = result.get("processing_summary", {})

    return {
        # Original / context
        "original_title": title,
        "original_description": description[:500],
        "original_subject": subject,
        "original_theme": theme,
        "original_keywords": keywords,
        "metadata_input": metadata_content,

        # Generated outputs
        "generated_keywords": json.dumps(result.get("generated_keywords", [])),
        "sponsored_keywords": json.dumps(result.get("sponsored_keywords", [])),
        "sector": result.get("sector", "Others"),

        # Processing metadata
        "keyword_logic": processing_summary.get("keyword_logic", ""),
        "sector_logic": processing_summary.get("sector_logic", ""),
        "sponsored_keywords_logic": processing_summary.get("sponsored_keywords", ""),
        "source_fields_used": json.dumps(result.get("source_fields_used", [])),
        "llm_response": json.dumps(result)
    }


# --------------------------------------------------------------------
# Setup CSV file
output_file = "results_catalog_keywords.csv"
csv_lock = Lock()  # Thread-safe file writes

fieldnames = [
    # Originals / context
    "original_title",
    "original_description",
    "original_subject",
    "original_theme",
    "original_keywords",
    "metadata_input",

    # Generated
    "generated_keywords",
    "sponsored_keywords",
    "sector",

    # Summary/debug
    "keyword_logic",
    "sector_logic",
    "sponsored_keywords_logic",
    "source_fields_used",
    "llm_response"
]

with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

# Parallel processing
results_array = []
max_workers = 8
rows_to_process = df.head(102)   # change to df for full run

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_keyword, row): idx
        for idx, row in rows_to_process.iterrows()
    }

    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()
            results_array.append(result_obj)

            # Thread-safe incremental write
            with csv_lock:
                with open(output_file, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writerow(result_obj)

            print(
                f"TITLE: {result_obj['original_title']}\n"
                f"SECTOR: {result_obj['sector']}\n"
                f"KEYWORDS: {result_obj['generated_keywords']}\n"
                f"SPONSORED: {result_obj['sponsored_keywords']}\n\n✓ Written to CSV"
            )
            print("\n----------------------------\n")

        except Exception as e:
            logging.error(f"Row processing failed: {e}")

print(f"\n✓ Complete! Results saved to {output_file}")


2025-11-24 11:43:38,847 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:43:38,855 : INFO - Result is {'generated_keywords': ['RajyaSabha', 'QuestionsAnswers', 'Session260', 'ParliamentQueries', 'RajyaSabhaQA', 'ParliamentaryData', 'Session260QA', 'AnnexuresData'], 'sponsored_keywords': ['RajyaSabhaQuestions', 'Session260', 'ParliamentQA', 'QuestionsData'], 'sector': 'Governance & Administration', 'processing_summary': {'keyword_logic': 'Derived from the catalog title and description to create 6–10 layman-friendly CamelCase terms that reflect Rajya Sabha Q&A content, sessions, and related data while avoiding department names and overly generic terms.', 'sector_logic': 'The catalog centers on parliamentary questions/answers data, aligning with Governance & Administration as the primary domain; reflects the overall governance-oriented nature of Rajya Sabha proceedings.', 'sponsored_keywords': 'Chosen 3–5 terms that most closely map to

TITLE: Answers Data of Rajya Sabha Questions for Session 260
SECTOR: Governance & Administration
KEYWORDS: ["RajyaSabha", "QuestionsAnswers", "Session260", "ParliamentQueries", "RajyaSabhaQA", "ParliamentaryData", "Session260QA", "AnnexuresData"]
SPONSORED: ["RajyaSabhaQuestions", "Session260", "ParliamentQA", "QuestionsData"]

✓ Written to CSV

----------------------------



2025-11-24 11:43:39,271 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:43:39,274 : INFO - Result is {'generated_keywords': ['PincodeBoundary', 'PostalBoundary', 'PostOfficeBoundary', 'PincodeMap', 'BoundaryGeoJson', 'PostalCodeData', 'PincodeDataset', 'PincodeLayer'], 'sponsored_keywords': ['GeoJsonBoundary', 'PincodeGeoJson', 'PostalBoundaryMap', 'PincodeData'], 'sector': 'Governance & Administration', 'processing_summary': {'keyword_logic': 'Derived 6–10 layman keywords from the catalog title and description focusing on Pincode boundaries, post office boundaries, and GeoJSON representation; used camelCase and kept terms concise (1–2 words).', 'sector_logic': "Selected Governance & Administration as the primary sector reflecting the catalog's administrative-geospatial nature related to postal boundaries and office locations.", 'sponsored_keywords': 'Chosen 3–4 terms that closely reflect the exact title components and data format:

TITLE: All India Pincode Boundary Geo JSON
SECTOR: Governance & Administration
KEYWORDS: ["PincodeBoundary", "PostalBoundary", "PostOfficeBoundary", "PincodeMap", "BoundaryGeoJson", "PostalCodeData", "PincodeDataset", "PincodeLayer"]
SPONSORED: ["GeoJsonBoundary", "PincodeGeoJson", "PostalBoundaryMap", "PincodeData"]

✓ Written to CSV

----------------------------



2025-11-24 11:43:40,069 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:43:40,074 : INFO - Result is {'generated_keywords': ['PerformanceGrading', 'GradingIndex', 'EducationMetrics', 'EducationAssessment', 'SchoolPerformance', 'EducationAnalytics', 'LearningOutcomes', 'EducationQuality'], 'sponsored_keywords': ['PGI', 'PerformanceIndex', 'EducationIndex', 'GradingMetrics'], 'sector': 'Education', 'processing_summary': {'keyword_logic': 'Derived 8 layman-friendly keywords (CamelCase, 1-2 words) from the catalog title/description and domain context. Prioritized terms that reflect performance grading, index concepts, and education outcomes while avoiding administrative terms.', 'sector_logic': "Chose Education as primary sector because the PGI pertains to educational performance grading and district-level education assessment, aligning with the catalog's focus.", 'sponsored_keywords': "Selected 4 near-title terms that closely map to '

...

TITLE: Import of Major Chemicals - Product-wise / Group-wise
SECTOR: Commerce & Industry
KEYWORDS: ["MajorChemicals", "ChemicalImports", "ProductWise", "GroupWise", "ImportQuantities", "ImportValues", "AlkaliChemicals", "DyesDyestuff", "InorganicChemicals", "OrganicChemicals"]
SPONSORED: ["MajorImports", "ChemImport", "ProductBreakdown", "GroupBreakdown"]

✓ Written to CSV

----------------------------



2025-11-24 11:49:44,839 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:49:44,845 : INFO - Result is {'generated_keywords': ['RealGDP', 'QuarterlyGDP', 'ConstantPrices', 'FinalConsumption', 'ConsumptionExpenditure', 'MacroEconomy', 'EconomicGrowth', 'Exports', 'Imports'], 'sponsored_keywords': ['QuarterlyGDP', 'RealGDP', 'GDPEstimates', 'ConstantPricesGDP'], 'sector': 'Finance & Economy', 'processing_summary': {'keyword_logic': 'Keywords were derived from the catalog’s focus on GDP, quarterly estimates, and macroeconomic aggregates at constant prices. They use CamelCase, are 1–2 words where possible, and avoid administrative terms. The set targets layman discoverability while reflecting the dataset content (GDP, consumption, exports/imports, macroeconomy).', 'sector_logic': 'Finance & Economy is identified as the primary sector since the catalog centers on GDP and macroeconomic indicators, aligning with economic governance and fisc

TITLE: Quarterly Estimates of GDP at Constant Prices
SECTOR: Finance & Economy
KEYWORDS: ["RealGDP", "QuarterlyGDP", "ConstantPrices", "FinalConsumption", "ConsumptionExpenditure", "MacroEconomy", "EconomicGrowth", "Exports", "Imports"]
SPONSORED: ["QuarterlyGDP", "RealGDP", "GDPEstimates", "ConstantPricesGDP"]

✓ Written to CSV

----------------------------



2025-11-24 11:49:51,123 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-24 11:49:51,130 : INFO - Result is {'generated_keywords': ['QuarterlyGDP', 'RealGDP', 'ConstantPrices', 'BaseYearPrices', 'GDPEstimates', 'MacroEconomy', 'EconomicIndicator', 'EconomicActivity'], 'sponsored_keywords': ['GDPQuarterly', 'NationalAccounts', 'EconomicGrowth', 'GDPOutput'], 'sector': 'Finance & Economy', 'processing_summary': {'keyword_logic': 'Derived 8 CamelCase keywords by extracting core concepts from the catalog title and description (GDP, quarterly, constant prices) and supplementing with related economic terms present in the subject/theme. The terms are kept to 1–2 words where possible, avoid administrative or India-specific labels, and are intended to improve layman discoverability of GDP-related quarterly data at constant prices.', 'sector_logic': 'Primary sector selected is Finance & Economy because the catalog centers on macroeconomic performan

TITLE: Quarterly Estimates of GDP at Constant Prices
SECTOR: Finance & Economy
KEYWORDS: ["QuarterlyGDP", "RealGDP", "ConstantPrices", "BaseYearPrices", "GDPEstimates", "MacroEconomy", "EconomicIndicator", "EconomicActivity"]
SPONSORED: ["GDPQuarterly", "NationalAccounts", "EconomicGrowth", "GDPOutput"]

✓ Written to CSV

----------------------------


✓ Complete! Results saved to results_catalog_keywords.csv


In [36]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import csv, json, logging

def process_row_keyword(row):
    """Process a single row - thread-safe"""
    title = row.get("title", "")
    description = row.get("description", "")
    catalog_title = row.get("catalog_title", "")
    ministry_department = row.get("ministry_department", "")
    sector = row.get("sector", "")
    sector_resource = row.get("sector_resource", "")
    cdos_state_ministry = row.get("cdos_state_ministry", "")
    note = row.get("note", "")

    
    # Fallback to 0 if missing / NaN
    raw_hvd = row.get("field_high_value_dataset", 0)
    try:
        hvd_flag = int(raw_hvd) if raw_hvd == raw_hvd else 0  # handles NaN
    except (ValueError, TypeError):
        hvd_flag = 0

    metadata_content = f"""
Title: {title}
Catalog Title: {catalog_title}
Ministry/Department: {ministry_department}
Sector: {sector}
Sector Resource: {sector_resource}
CDOS State Ministry: {cdos_state_ministry}
Note: {note}
HVD Flag: {hvd_flag}
""".strip()

    # LLM must now return JSON including "hvd_category"
    result = get_keywords(metadata_content)  # expected to return a dict (JSON-like)
    logging.info(f"Result is {result}")

    return {
        "title": title,
        "sector": sector,
        "hvd": hvd_flag,
        "metadata_input": metadata_content,
        "llm_response": json.dumps(result, ensure_ascii=False),
        "generated_keywords": result.get("enhanced_keywords", ""),
        "generated_sponsored_keywords": result.get("sponsored_keywords", ""),
        "generated_subject": result.get("generated_subject", ""),
        "generated_theme": result.get("generated_theme", ""),
        "justification": result.get("justification", ""),
        "confidence_score": result.get("confidence_score", 0),
        "metadata_gaps": json.dumps(result.get("metadata_gaps", []), ensure_ascii=False),
        # NEW: HVD classification coming from the LLM
        "hvd_category": result.get("hvd_category", ""),
    }

# CSV setup
output_file = "results_100.csv"
fieldnames = [
    "title",
    "sector",
    "hvd",             
    "metadata_input",
    "llm_response",
    "generated_keywords",
    "generated_sponsored_keywords",
    "generated_subject",
    "generated_theme",
    "justification",
    "confidence_score",
    "metadata_gaps",
    "hvd_category",    
]

csv_lock = Lock()
with open(output_file, "w", newline="", encoding="utf-8") as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

results_array = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_row = {
        executor.submit(process_row_keyword, row): idx
        for idx, row in (df.head(102)).iterrows()
    }
    for future in as_completed(future_to_row):
        try:
            result_obj = future.result()
            results_array.append(result_obj)

            # Thread-safe incremental write
            with csv_lock:
                with open(output_file, "a", newline="", encoding="utf-8") as f:
                    csv.DictWriter(f, fieldnames=fieldnames).writerow(result_obj)

            print(
                f"TITLE: {result_obj['title']}\n"
                f"SECTOR: {result_obj['sector']}\n"
                f"HVD: {result_obj['hvd']}\n"
                f"HVD CATEGORY: {result_obj['hvd_category']}\n\nWritten to CSV"
            )


            print("\n----------------------------\n")

        except Exception as e:
            logging.error(f"Row processing failed: {e}")

print(f"\nComplete Results saved to {output_file}")


2025-11-15 07:19:06,296 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-15 07:19:06,301 : INFO - Result is {'title': 'List of MSME Registered Units under UDYAM', 'enhanced_keywords': 'MSMRegistration, RegisteredUnits, UDYAMList, UDYAMRegistration, MSMEsData, UnitRegistration, MSMEInfo, IndustryRegistrations', 'sponsored_keywords': 'MSMRegisteredUnits, UDYAMRegistration, RegisteredUnits, MSMERegistration', 'justification': 'Keywords reflect the dataset content of MSME units registered under UDYAM; the dataset is a registry of units, aligning with the statistics category due to structured, count-like data.', 'confidence_score': 0.84, 'metadata_gaps': ['Geographic coverage not specified', 'Temporal coverage not specified', 'Fields included in the dataset not described', 'Source organization details not explicit'], 'hvd_category': 'statistics'}


TITLE: List of MSME Registered Units under UDYAM
SECTOR: Industries;Medium;Micro;Small Scale
HVD: 1
HVD CATEGORY: statistics

Written to CSV

----------------------------



2025-11-15 07:19:09,649 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-15 07:19:09,656 : INFO - Result is {'title': 'Kisan Call Centre (KCC) - Transcripts of farmers queries & answers', 'enhanced_keywords': 'KisanCall, FarmersQueries, DistrictQueries, MonthWise, QueryResponses, FarmerSupport, AgricultureHelpdesk, KccData, TranscriptData, CallCentre', 'sponsored_keywords': 'KisanCallCentre, FarmersQueries, DistrictQueries, MonthWise', 'justification': 'Keywords reflect the district-wise and month-wise farmer queries and call-center responses. The dataset is best categorized under statistics due to its structured, district- and month-level query data.', 'confidence_score': 0.82, 'metadata_gaps': ['Temporal coverage specifics (start/end dates) not provided', 'Geographic coverage across districts/states not clearly defined', 'Data format and accessibility of transcripts (plain text vs. structured) not specified', 'Language(s) of transcripts

TITLE: Kisan Call Centre (KCC) - Transcripts of farmers queries & answers
SECTOR: Agriculture
HVD: 1
HVD CATEGORY: statistics

Written to CSV

----------------------------



2025-11-15 07:19:11,807 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-15 07:19:11,809 : INFO - Result is {'title': 'Current Daily Price of Various Commodities from Various Markets (Mandi)', 'enhanced_keywords': 'CommodityPrices, MandiMarkets, MarketPrices, DailyPrices, AgriculturePrices, CropPrices, PriceTrends, FarmersMarkets', 'sponsored_keywords': 'MandiPrices, CurrentPrices, CommodityPrices, MarketPrices', 'justification': 'Keywords reflect current daily commodity pricing across markets. The dataset provides structured numeric market price data, justifying a statistics classification.', 'confidence_score': 0.82, 'metadata_gaps': ['Geographic coverage not specified', 'Temporal coverage start/date not specified', 'Price unit not specified', 'Data collection methodology not described'], 'hvd_category': 'statistics'}


...

TITLE: Boundaries of Agro-climatic regions
SECTOR: Ground Water;Surface Water
HVD: 1
HVD CATEGORY: geospatial

Written to CSV

----------------------------



2025-11-15 07:23:33,717 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-15 07:23:33,719 : INFO - Result is {'title': 'State-wise Population, Decadal Population Growth rate and Population Density - 2011', 'enhanced_keywords': 'StatePopulation, PopulationDensity, DecadalGrowth, Population2011, RuralHealth, CensusData, StateWiseStats, Demographics', 'sponsored_keywords': 'GrowthRate, Census2011, StateDensity, PopulationTrend', 'justification': 'The keywords map to the core contents: state-level population, density, and decadal growth for 2011 as described in the title. The dataset is part of Rural Health Statistics; no high-value data designation is assigned (HVD Flag = 0).', 'confidence_score': 0.78, 'metadata_gaps': ['Geographic scope details missing beyond states', 'Temporal coverage limited to 2011; no trend data', 'Data sources and methodology not fully documented in Note'], 'hvd_category': ''}


TITLE: State-wise Population, Decadal Population Growth rate and Population Density - 2011
SECTOR: Health and Family welfare;Family Welfare;Health
HVD: 0
HVD CATEGORY: 

Written to CSV

----------------------------



2025-11-15 07:23:36,362 : INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-15 07:23:36,364 : INFO - Result is {'title': 'State-wise Details of the Infant Mortality Rate, Institutional Delivery and Prevalence of Under Weight Children under five years of Age (in Reply to Unstarred Question on 07 December, 2022)', 'enhanced_keywords': 'InfantMortality, InstitutionalDelivery, UnderFive, StateWiseDetails, ChildNutrition, WestBengal, HealthStatus, RegionalVariation', 'sponsored_keywords': 'InfantMortality, StateWiseDetails, InstitutionalDelivery, UnderFive', 'justification': "Keywords reflect the dataset's focus on state-wise infant mortality, delivery practices, and nutritional status of children under five, as indicated by the title and note. The terms are user-friendly and non-technical, aiding discoverability while staying aligned with the dataset content; the HVD flag is 0, so no HV data category is assigned.", 'confidence_score': 0.72, 'met

TITLE: State-wise Details of the Infant Mortality Rate, Institutional Delivery and Prevalence of Under Weight Children under five years of Age (in Reply to Unstarred Question on 07 December, 2022)
SECTOR: All
HVD: 0
HVD CATEGORY: 

Written to CSV

----------------------------


Complete Results saved to results_100.csv


ValueError: Categorical categories must be unique

In [37]:
result_df=pd.read_csv("/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/notebooks/results_100.csv")
result_df.count(axis=0)

title                           100
sector                          100
hvd                             100
metadata_input                  100
llm_response                    100
generated_keywords              100
generated_sponsored_keywords    100
generated_subject                 0
generated_theme                   0
justification                   100
confidence_score                100
metadata_gaps                   100
hvd_category                     42
dtype: int64

In [15]:
results_array

[{'title': 'Current Daily Price of Various Commodities from Various Markets (Mandi)',
  'sector': 'Agriculture;Agricultural Marketing',
  'metadata_input': 'Title: Current Daily Price of Various Commodities from Various Markets (Mandi)\nCatalog Title: Current daily price of various commodities from various markets (Mandi)\nMinistry/Department: Ministry of Agriculture and Farmers Welfare;Department of Agriculture and Farmers Welfare;Directorate of Marketing and Inspection (DMI)\nSector: Agriculture;Agricultural Marketing\nSector Resource: Agriculture;Agricultural Marketing\nCDOS State Ministry: Directorate of Marketing and Inspection (DMI)\nNote: nan',
  'llm_response': '{"Title": "Current Daily Price of Various Commodities from Various Markets (Mandi)"}',
  'generated_keywords': '',
  'generated_subject': '',
  'generated_theme': '',
  'justification': '',
  'confidence_score': 0,
  'metadata_gaps': '[]'},
 {'title': 'Registrars of Companies (RoC)-wise Company Master Data',
  'sector':

In [19]:
with open('/home/aakash/NIC/cdl-updated/nic-metadata-cleaning/notebooks/metadata_enrichment_result.json') as f:
    json.dump(results_array, f, indent=2, ensure_ascii=False)

print(f"\n Saved {len(results_array)} results to metadata_enrichment_results.json")

UnsupportedOperation: not writable